# Ajuste de YOLO para Deteccion de Eventos Sonoros (YOHO)

## Por que YOLO para audio

Nuestro proyecto reformula la Deteccion de Eventos Sonoros (SED) como un problema de deteccion de objetos en el plano tiempo-frecuencia. En lugar de tratar el audio como una secuencia temporal pura, lo convertimos en una imagen (espectrograma) donde cada evento sonoro es un objeto que ocupa una region especifica.

Esta aproximacion esta respaldada por literatura reciente: trabajos como el de Gonzalez et al. (2026) demuestran que adaptar detectores modernos como YOLOv8 a espectrogramas mel es competitivo frente a arquitecturas tradicionales como CRNN o Conformer, con ventajas en condiciones de solapamiento y ruido. Ademas, el propio nombre YOHO (You Only Hear Once) es una reinterpretacion deliberada del paradigma single-pass de YOLO: procesamos la representacion visual del audio en una sola pasada para detectar eventos.

## Que necesitamos ajustar

YOLO esta disenado para imagenes RGB de 3 canales, pero nuestro dataset usa espectrogramas en escala de grises de 1 canal (PNG de 16 bits, modo I;16). Necesitamos indicarle a YOLO que trabaje con `channels: 1`. Ultralytics anadio soporte oficial para esto en versiones recientes (v8.3.131+), con un parametro `channels` en el data loader y la posibilidad de declararlo directamente en el `data.yaml`. La documentacion oficial confirma que basta con anadir `channels: 1` al YAML para que YOLO convierta las imagenes a escala de grises durante el entrenamiento.

## Que configuracion usaremos

| Parametro | Valor | Justificacion |
|---|---|---|
| Modelo | `yolo11n.pt` (o `yolov8n.pt`) | Modelo nano, rapido, adecuado para validar el pipeline |
| `imgsz` | 1024 | Nuestras imagenes miden aproximadamente 1032x1024 px; 1024 es multiplo de 32 y el mas cercano |
| `epochs` | 100 | Estandar; con `patience` se detiene antes si no mejora |
| `batch` | -1 (AutoBatch) | Ultralytics ajusta automaticamente el maximo que cabe en 8 GB VRAM |
| `patience` | 30 | Early stopping para ahorrar tiempo |
| `device` | 0 | Usa la RTX 5060 |
| `channels` | 1 | Ya esta en el `data.yaml` generado |

In [ ]:
import sys
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
from collections import Counter

sys.path.append("../src")
from config import (
    DATASET_ROOT,
    AUDIO_DIR,
    CSV_MONO,
    OUTPUT_ROOT,
    SPECTROGRAM_DIR,
    LABELS_DIR,
)

In [ ]:
# Verificar que el data.yaml existe y mostrar su contenido
data_yaml_path = OUTPUT_ROOT / "data.yaml"
print(f"Ruta del data.yaml: {data_yaml_path}")
print(f"Existe: {data_yaml_path.exists()}")
print("\n--- Contenido del data.yaml ---")
with open(data_yaml_path, 'r') as f:
    print(f.read())

## Sobre el `channels: 1` y los pesos preentrenados

Cuando usamos `channels: 1`, Ultralytics reinicia los pesos de la primera capa convolucional (la que recibe la imagen), porque los pesos originales de `yolo11n.pt` esperan 3 canales RGB. El resto del modelo (backbone, neck, head) conserva los pesos preentrenados en COCO. Esto es normal y esperado: la red aprendera a extraer caracteristicas de nuestros espectrogramas en esa primera capa durante el entrenamiento.

En el informe podemos mencionarlo como una decision de diseno: "Se utilizo el soporte nativo de Ultralytics para entrada de 1 canal (`channels: 1`), lo que implica que la primera capa convolucional se inicializa aleatoriamente mientras el resto del modelo conserva los pesos preentrenados en COCO."

In [ ]:
splits = ['train', 'val', 'test']
for split in splits:
    img_dir = OUTPUT_ROOT / split
    if not img_dir.exists():
        print(f"{split}: carpeta no encontrada")
        continue
    imgs = list(img_dir.glob("*.png"))
    if not imgs:
        print(f"{split}: sin imagenes")
        continue
    modos = Counter(Image.open(p).mode for p in imgs[:20])
    print(f"{split}: {len(imgs)} imagenes | Modos (primeras 20): {dict(modos)}")

In [ ]:
train_script = '''
from ultralytics import YOLO
from pathlib import Path
import sys

sys.path.append(str(Path(__file__).resolve().parent))
from config import OUTPUT_ROOT

def main():
    data_yaml = OUTPUT_ROOT / "data.yaml"
    model = YOLO("yolo11n.pt")

    results = model.train(
        data=str(data_yaml),
        epochs=100,
        imgsz=1024,
        batch=-1,
        patience=30,
        workers=4,
        device=0,
        project=str(OUTPUT_ROOT / "runs"),
        name="yoho_train",
        exist_ok=True,
        pretrained=True,
        verbose=True,
    )
    print("Entrenamiento finalizado.")
    print(f"Resultados en: {results.save_dir}")

if __name__ == "__main__":
    main()
'''

train_path = Path("../src/train.py")
train_path.write_text(train_script)
print(f"Script guardado en: {train_path}")
print("\n--- Contenido de train.py ---")
print(train_script)

## Consideraciones antes de lanzar el entrenamiento

1. Tamano del dataset: aproximadamente 3,541 ventanas (tras el split, aproximadamente 2,833 train, 354 val, 354 test). Es un tamano manejable para una GPU de 8 GB.
2. Ventanas sin cajas: El script de generacion ya creo archivos `.txt` vacios para ventanas sin eventos. YOLO las tratara como ejemplos negativos (fondo), lo que ayuda a reducir falsos positivos.
3. Metricas a revisar: Tras el entrenamiento, revisaremos `mAP@0.5`, `mAP@0.5:0.95`, la matriz de confusion y las curvas Precision-Recall. Prestaremos especial atencion a las clases con menos muestras (posible desbalance) y a las de eventos cortos.
4. Posibles ajustes: Si el entrenamiento se estanca o hay sobreajuste, podremos ajustar `patience`, `batch`, o probar data augmentation especifica para audio (ruido, desplazamiento temporal).

## Siguiente paso

Ejecutar `python src/train.py` desde la terminal y monitorear el progreso. Una vez finalizado, analizaremos las metricas en el notebook `03_resultados.ipynb`.

In [ ]:
import cv2
from pathlib import Path

corruptos = []
for p in Path("/mnt/f/DATASETS/DataSED/yolo_dataset/images").rglob("*.png"):
    img = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    if img is None:
        corruptos.append(p)

print(f"Archivos que OpenCV no puede leer: {len(corruptos)}")
for p in corruptos:
    print(p)

# RESULTADO

`(aexxx) aexxtool@AExxTool:~/YOHO-by-AEx/src$ python3 train.py
New https://pypi.org/project/ultralytics/8.4.208 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.205 🚀 Python-3.12.3 torch-2.15.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/mnt/f/DATASETS/DataSED/yolo_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yoho_train, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=30, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=/mnt/f/DATASETS/DataSED/yolo_dataset/runs, quantize=None, rect=False, resume=False, retina_masks=False, rle=1.0, save=True, save_conf=False, save_crop=False, save_dir=/mnt/f/DATASETS/DataSED/yolo_dataset/runs/yoho_train, save_frames=False, save_json=False, save_period=-1, save_txt=False, scale=0.5, seed=0, shear=0.0, show=False, show_boxes=True, show_conf=True, show_labels=True, simplify=True, single_cls=False, source=None, split=val, stream_buffer=False, task=detect, tile=False, time=None, tracker=tracktrack.yaml, translate=0.1, val=True, verbose=True, vid_stride=1, visualize=False, warmup_bias_lr=0.1, warmup_epochs=3.0, warmup_momentum=0.8, weight_decay=0.0005, workers=4, workspace=None
Overriding model.yaml nc=80 with nc=22

                   from  n    params  module                                       arguments
  0                  -1  1       176  ultralytics.nn.modules.conv.Conv             [1, 16, 3, 2]
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]
  8                  -1  1    346112  ultralytics.nn.modules.block.C3k2            [256, 256, 1, True]
  9                  -1  1    164608  ultralytics.nn.modules.block.SPPF            [256, 256, 5]
 10                  -1  1    249728  ultralytics.nn.modules.block.C2PSA           [256, 256, 1]
 11                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']
 12             [-1, 6]  1         0  ultralytics.nn.modules.conv.Concat           [1]
 13                  -1  1    111296  ultralytics.nn.modules.block.C3k2            [384, 128, 1, False]
 14                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']
 15             [-1, 4]  1         0  ultralytics.nn.modules.conv.Concat           [1]
 16                  -1  1     32096  ultralytics.nn.modules.block.C3k2            [256, 64, 1, False]
 17                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]
 18            [-1, 13]  1         0  ultralytics.nn.modules.conv.Concat           [1]
 19                  -1  1     86720  ultralytics.nn.modules.block.C3k2            [192, 128, 1, False]
 20                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]
 21            [-1, 10]  1         0  ultralytics.nn.modules.conv.Concat           [1]
 22                  -1  1    378880  ultralytics.nn.modules.block.C3k2            [384, 256, 1, True]
 23        [16, 19, 22]  1    434962  ultralytics.nn.modules.head.Detect           [22, 16, None, [64, 128, 256]]
YOLO11n summary: 181 layers, 2,593,842 parameters, 2,593,826 gradients, 6.5 GFLOPs

Remapped 1/22 cls head rows from pretrained weights by class name
Transferred 499/499 items from pretrained weights
Freezing layer 'model.23.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 2.3±0.5 ms, read: 102.4±110.5 MB/s, size: 1451.8 KB)
train: Scanning /mnt/f/DATASETS/DataSED/yolo_dataset/labels/train.cache... 2848 images, 76 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2848/2848 597.3Mit/s 0.0s
AutoBatch: Computing optimal batch size for imgsz=1024 at 60.0% GPU memory utilization.
AutoBatch: CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU) 7.96G total, 0.12G reserved, 6.82G available
      Params      GFLOPs GPU_mem (GiB)  forward (ms) backward (ms)                   input                  output
     2609698       16.98         0.413         315.9         653.7      (1, 1, 1024, 1024)                    list
     2609698       33.96         0.807         141.4         741.8      (2, 1, 1024, 1024)                    list
     2609698       67.91         1.599         225.1         508.7      (4, 1, 1024, 1024)                    list
     2609698       135.8         3.137         217.2         785.6      (8, 1, 1024, 1024)                    list
     2609698       271.7         6.278         208.7         935.4     (16, 1, 1024, 1024)                    list
AutoBatch: Using batch-size 10 for CUDA:0 3.93G/6.82G (58%) ✅
train: Fast image access ✅ (ping: 6.5±1.4 ms, read: 55.3±23.8 MB/s, size: 1207.8 KB)
train: Scanning /mnt/f/DATASETS/DataSED/yolo_dataset/labels/train.cache... 2848 images, 76 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2848/2848 853.2Mit/s 0.0s
WARNING ⚠️ val: Slow image access detected (ping: 2.6±0.6 ms, read: 22.0±3.1 MB/s, size: 1258.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /mnt/f/DATASETS/DataSED/yolo_dataset/labels/val.cache... 346 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 346/346 181.4Mit/s 0.0s
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and determining best 'optimizer' and 'lr0' automatically...
optimizer: AdamW(lr=0.000385, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.00046875), 87 bias(decay=0.0)
Plotting labels to /mnt/f/DATASETS/DataSED/yolo_dataset/runs/yoho_train/labels.jpg...
Using 2848 train, 346 val images for fraction=1.0 at imgsz=1024
Using 4 dataloader workers
Logging results to /mnt/f/DATASETS/DataSED/yolo_dataset/runs/yoho_train
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      3.66G      1.792      5.031      1.958         36       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 1.3it/s 13.9s
                   all        346        836       0.41      0.155     0.0854     0.0337

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100      3.67G      1.168      2.791      1.455         46       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.1it/s 5.8s
                   all        346        836       0.23      0.184      0.124     0.0579

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      3.67G      1.083      2.525      1.398         30       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.5s
                   all        346        836      0.387      0.249      0.213      0.112

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      3.66G     0.9787      2.372      1.313         34       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.5s
                   all        346        836      0.378      0.298      0.271      0.162

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      5/100      3.66G     0.9265      2.214      1.257         39       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.5s
                   all        346        836      0.621      0.314      0.338      0.209

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      6/100      3.67G     0.8978      2.145      1.236         47       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:41
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.609      0.313      0.331      0.193

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      7/100      3.66G     0.8843      2.063      1.211         58       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.441       0.35       0.34      0.201

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      8/100      3.66G     0.8676      1.993        1.2         40       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.4s
                   all        346        836      0.442       0.41      0.402      0.242

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      9/100      3.67G     0.8514      1.952      1.187         56       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.387      0.399      0.377       0.23

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     10/100      3.66G     0.8523      1.925      1.187         40       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.641      0.372      0.437      0.261

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     11/100      3.66G     0.8342      1.895      1.175         38       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.618      0.371      0.407      0.235

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     12/100      3.67G     0.8216      1.842       1.17         43       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836       0.48      0.367      0.396      0.244

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     13/100      3.66G     0.8233       1.83      1.162         52       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.484       0.45       0.44      0.267

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     14/100      3.66G     0.7973      1.815      1.165         36       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.628      0.405      0.473       0.29

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     15/100      3.67G     0.7986      1.763      1.155         33       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836       0.56      0.455      0.491      0.299

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     16/100      3.66G     0.7912      1.717      1.152         50       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.673      0.468      0.496      0.311

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     17/100      3.67G      0.776      1.664      1.144         43       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.595      0.441      0.488      0.303

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     18/100      3.66G     0.7691      1.656      1.142         54       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.619      0.464      0.485      0.306

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     19/100      3.66G     0.7789      1.639       1.14         44       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.5s
                   all        346        836      0.586      0.445      0.471      0.269

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     20/100      3.66G     0.7709      1.639      1.138         32       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.5s
                   all        346        836      0.638      0.465      0.535      0.336

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     21/100      3.67G      0.773      1.615      1.129         48       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.687       0.46      0.526      0.335

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     22/100      3.67G     0.7702      1.607      1.132         49       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.626      0.467      0.533      0.333

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     23/100      3.66G     0.7481      1.575      1.117         34       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.639      0.493      0.539      0.335

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     24/100      3.66G     0.7556      1.562      1.125         30       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.653      0.472      0.522       0.33

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     25/100      3.66G     0.7537      1.567      1.124         48       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.598      0.457      0.512      0.318

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     26/100      3.67G     0.7293      1.534       1.11         30       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.634      0.512      0.552      0.333

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     27/100      3.67G     0.7518      1.526      1.122         45       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.724       0.52      0.563      0.358

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     28/100      3.66G     0.7187      1.505      1.119         39       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.643      0.532      0.566      0.354

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     29/100      3.67G     0.7331       1.49      1.117         30       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.669      0.505      0.583       0.37

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     30/100      3.67G     0.7264      1.454      1.113         43       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.686      0.488      0.541      0.337

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     31/100      3.67G      0.728      1.493      1.113         38       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.665      0.545      0.586      0.369

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     32/100      3.67G     0.7093      1.438      1.105         34       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836       0.71      0.511      0.552      0.351

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     33/100      3.66G     0.7218      1.459      1.107         40       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.4s
                   all        346        836      0.733      0.502      0.573      0.355

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     34/100      3.66G     0.7171      1.453      1.108         43       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.724      0.519      0.589      0.363

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     35/100      3.66G     0.7063      1.406      1.103         40       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.718      0.493      0.578      0.363

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     36/100      3.66G     0.7147      1.422      1.106         32       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.681      0.534      0.585      0.369

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     37/100      3.67G     0.7022      1.405      1.093         28       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.763      0.514      0.601      0.391

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     38/100      3.66G     0.7091      1.393      1.096         44       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.689      0.513      0.577      0.362

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     39/100      3.66G     0.7076      1.385      1.093         45       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836       0.68      0.539      0.591      0.389

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     40/100      3.66G     0.7046      1.371      1.099         33       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.781      0.544      0.623      0.403

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     41/100      3.66G     0.6829      1.363      1.097         48       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.806       0.51      0.606      0.395

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     42/100      3.66G     0.6884      1.324      1.085         51       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.677      0.552      0.603      0.385

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     43/100      3.67G     0.6868      1.338      1.083         46       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.706      0.529      0.592      0.374

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     44/100      3.67G     0.6891       1.32      1.086         38       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.744       0.53      0.592       0.38

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     45/100      3.66G     0.6873      1.344       1.09         45       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.738      0.516      0.591      0.379

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     46/100      3.67G     0.6831      1.333      1.078         49       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.615       0.57      0.601      0.389

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     47/100      3.66G     0.6805      1.336      1.082         36       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.4s
                   all        346        836      0.695      0.567       0.63      0.411

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     48/100      3.67G     0.6832      1.313      1.082         45       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.728      0.534      0.613      0.393

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     49/100      3.66G     0.6778      1.313      1.088         56       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.761      0.544      0.634      0.407

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     50/100      3.67G     0.6804      1.274      1.083         39       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.669      0.578      0.622      0.404

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     51/100      3.66G     0.6681      1.267      1.079         44       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.767       0.57      0.636      0.416

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     52/100      3.67G     0.6634      1.255       1.07         53       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.778      0.555      0.643      0.416

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     53/100      3.66G     0.6649      1.258      1.071         43       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.722       0.57      0.636      0.415

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     54/100      3.66G     0.6638      1.259      1.071         44       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.737      0.566      0.626      0.409

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     55/100      3.66G     0.6612      1.245      1.076         44       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.792      0.573      0.649      0.427

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     56/100      3.66G      0.659      1.254      1.077         38       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.4s
                   all        346        836      0.781      0.559      0.636      0.427

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     57/100      3.66G      0.663      1.252      1.068         42       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836       0.75      0.562      0.637      0.425

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     58/100      3.66G      0.655      1.209      1.071         48       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.773      0.555      0.644      0.426

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     59/100      3.66G     0.6492      1.221      1.067         39       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.822      0.555      0.657       0.43

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     60/100      3.67G     0.6549      1.191       1.06         58       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.4s
                   all        346        836      0.767      0.563       0.63      0.412

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     61/100      3.66G     0.6533      1.197      1.066         59       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.781      0.574      0.649      0.424

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     62/100      3.66G     0.6544      1.208      1.072         41       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.803      0.541      0.636      0.418

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     63/100      3.66G     0.6528      1.221      1.066         49       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836       0.79      0.559      0.659      0.434

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     64/100      3.66G     0.6368      1.168       1.06         38       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.728      0.593       0.65       0.42

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     65/100      3.67G     0.6372       1.16      1.051         39       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.756      0.594      0.663       0.43

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     66/100      3.66G     0.6418      1.165      1.053         35       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.779       0.59      0.664      0.436

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     67/100      3.67G     0.6308      1.146      1.055         40       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836       0.72      0.552      0.639      0.425

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     68/100      3.66G     0.6451      1.171      1.059         49       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.733      0.579      0.658      0.434

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     69/100      3.67G     0.6354      1.158      1.059         42       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.766      0.573      0.663      0.435

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     70/100      3.66G     0.6341      1.152      1.055         48       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.719      0.613      0.675      0.449

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     71/100      3.66G     0.6325      1.114      1.053         56       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.744      0.582      0.643      0.424

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     72/100      3.67G     0.6237      1.131      1.052         37       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.4s
                   all        346        836        0.7        0.6      0.653      0.433

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     73/100      3.66G     0.6185      1.109      1.046         47       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.739      0.599      0.669      0.442

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     74/100      3.67G     0.6214      1.144      1.052         48       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.742      0.592      0.658      0.433

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     75/100      3.66G     0.6367      1.138      1.052         44       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.4s
                   all        346        836      0.766      0.599      0.677      0.446

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     76/100      3.67G     0.6259      1.122      1.051         37       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.757       0.61      0.683      0.455

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     77/100      3.66G     0.6271      1.104      1.045         41       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.4s
                   all        346        836      0.729       0.61      0.674      0.447

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     78/100      3.67G     0.6145      1.101      1.046         38       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.775      0.583      0.666      0.446

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     79/100      3.66G     0.6191       1.08      1.046         38       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.778      0.593      0.669      0.446

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     80/100      3.66G     0.6069      1.073      1.036         39       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.781       0.59      0.676      0.446

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     81/100      3.66G     0.6216      1.075      1.043         33       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.727      0.595      0.664       0.44

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     82/100      3.66G     0.6047      1.062      1.046         41       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.769      0.595      0.667      0.438

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     83/100      3.66G     0.6037      1.074      1.041         41       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.788      0.597      0.666      0.437

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     84/100      3.66G     0.5966      1.061      1.042         48       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.838      0.582      0.672      0.443

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     85/100      3.66G     0.6076      1.062      1.038         44       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.4s
                   all        346        836      0.774        0.6      0.667      0.439

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     86/100      3.66G     0.5994      1.052      1.034         63       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.4s
                   all        346        836      0.725      0.616      0.663      0.443

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     87/100      3.66G     0.6112      1.072      1.038         52       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.809      0.589      0.677      0.446

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     88/100      3.66G      0.601       1.05      1.038         46       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.767      0.614      0.672      0.443

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     89/100      3.66G     0.5977      1.025      1.041         49       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.7s
                   all        346        836      0.789      0.596      0.681      0.449

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     90/100      3.66G     0.6051       1.05      1.033         35       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.794       0.59      0.666      0.443
Closing dataloader mosaic

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     91/100      3.66G     0.8892      1.016     0.9384         19       1024: 100% ━━━━━━━━━━━━ 285/285 2.7it/s 1:45
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.781      0.592      0.659      0.434

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     92/100      3.66G     0.8462     0.9206     0.9018         17       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:41
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836       0.79      0.618       0.68      0.454

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     93/100      3.66G     0.8381     0.8943     0.9044         18       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:41
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.762      0.624      0.677      0.453

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     94/100      3.66G     0.8345     0.8914     0.8948         24       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:41
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.778      0.622      0.685       0.46

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     95/100      3.66G     0.8287     0.8693     0.8947         17       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.801      0.623       0.69      0.462

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     96/100      3.66G     0.8359     0.8726     0.8962         15       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:41
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.4s
                   all        346        836      0.787      0.632      0.689      0.463

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     97/100      3.66G     0.8283     0.8545     0.8864         17       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:41
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.2it/s 5.6s
                   all        346        836      0.816       0.61      0.686      0.462

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     98/100      3.66G     0.8255     0.8471     0.8826         11       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.759      0.648      0.694      0.466

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     99/100      3.66G     0.8306      0.862     0.8888         16       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.4s
                   all        346        836      0.811      0.609      0.691      0.463

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    100/100      3.66G     0.8179     0.8335     0.8892         13       1024: 100% ━━━━━━━━━━━━ 285/285 2.8it/s 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 3.3it/s 5.5s
                   all        346        836      0.763      0.629      0.689      0.462

100 epochs completed in 3.170 hours.
Optimizer stripped from /mnt/f/DATASETS/DataSED/yolo_dataset/runs/yoho_train/weights/last.pt, 5.6MB
Optimizer stripped from /mnt/f/DATASETS/DataSED/yolo_dataset/runs/yoho_train/weights/best.pt, 5.6MB

Validating /mnt/f/DATASETS/DataSED/yolo_dataset/runs/yoho_train/weights/best.pt...
Ultralytics 8.4.205 🚀 Python-3.12.3 torch-2.15.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
YOLO11n summary (fused): 101 layers, 2,601,866 parameters, 0 gradients, 6.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.4it/s 7.5s
                   all        346        836      0.756      0.651      0.694      0.467
                 Bells         13         25      0.921       0.88      0.886      0.646
                 Birds         25         35      0.604      0.175      0.232      0.142
   Cat fight and moans          7         21      0.773      0.809      0.851      0.547
          Chicken coop         12         32      0.685      0.612      0.582      0.307
  Cicadas and crickets         26         65       0.87       0.82      0.907      0.704
Crows seagulls and magpies         14         21      0.739      0.524      0.611      0.309
Dog barkings and howlings         17         31      0.702      0.645      0.693      0.394
        Glass breaking          6         15      0.692        0.8       0.83      0.455
                  Horn         15         29       0.77      0.346      0.475      0.233
         Jet aircrafts         23         25       0.57       0.28      0.283      0.113
Lawn mower brush cutter and olive shaker         25         34      0.863      0.618      0.717       0.55
                 Music         24         36      0.828      0.833      0.893      0.666
   Propeller aircrafts         34         54      0.823      0.741      0.794      0.603
     Sirens and alarms         20         25      0.779       0.76      0.802      0.559
Thunder fireworks and gunshot         15         22      0.695      0.682      0.676      0.421
                 Train         40         49      0.747      0.724      0.738      0.466
Vacuum cleaner fan and hairdryer         19         33      0.968      0.909      0.953      0.797
        Vehicle idling         29         36      0.501      0.306      0.442      0.289
       Vehicle pass-by         58         76      0.749      0.353      0.439      0.256
                Voices         31         55       0.55      0.636      0.555      0.291
          Wind turbine         29         38      0.863      0.921      0.936      0.712
              Workshop         37         79      0.949      0.944      0.975      0.812
Speed: 0.2ms preprocess, 17.5ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /mnt/f/DATASETS/DataSED/yolo_dataset/runs/yoho_train
Entrenamiento finalizado.
Resultados en: /mnt/f/DATASETS/DataSED/yolo_dataset/runs/yoho_train`